In [14]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# =========================
# 配置
# =========================
CSV_PATH = "/Users/colin/Downloads/queries_with_predicates - p2.csv"   # 改成你的实际路径
OUT_DIR = "figs"
os.makedirs(OUT_DIR, exist_ok=True)

# =========================
# 读数据
# =========================
df = pd.read_csv(CSV_PATH)

# 列名（按你这份 csv 的实际结构）
TRUE_COL = "truecard"
SAFE_QERR_COL = "qerror"        # safebound 的 qerror
GCARD_QERR_COLS = ["qerror.1", "qerror.2", "qerror.3"]
GCARD_EST_COLS  = ["GCard_1", "GCard_2", "GCard_3"]
INDEX_COL = "index"

def to_num(s):
    return pd.to_numeric(s, errors="coerce")

def safe_log10(series):
    s = to_num(series)
    s = s.where(s > 0)
    return np.log10(s)

# 衍生指标
df["gcard_qerr_mean"] = df[GCARD_QERR_COLS].apply(to_num).mean(axis=1)
df["gcard_qerr_std"]  = df[GCARD_QERR_COLS].apply(to_num).std(axis=1, ddof=0)
df["gcard_qerr_cv"]   = df["gcard_qerr_std"] / df["gcard_qerr_mean"].replace(0, np.nan)

df["gcard_est_mean"]  = df[GCARD_EST_COLS].apply(to_num).mean(axis=1)
df["gcard_est_std"]   = df[GCARD_EST_COLS].apply(to_num).std(axis=1, ddof=0)
df["gcard_est_cv"]    = df["gcard_est_std"] / df["gcard_est_mean"].replace(0, np.nan)

# =========================
# 1) qerror 箱线图：SafeBound vs (GCard qerror均值)
# =========================
plt.figure()
data = [
    to_num(df[SAFE_QERR_COL]).dropna(),
    to_num(df["gcard_qerr_mean"]).dropna(),
]
plt.boxplot(data, labels=["SafeBound qerror", "GCard(mean of 1~3) qerror"], showfliers=True)
plt.yscale("log")  # qerror 跨度大更直观；不想要可删
plt.ylabel("q-error (log scale)")
plt.title("Q-error Boxplot: SafeBound vs GCard(mean)")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "1_qerror_boxplot_safebound_vs_gcard_mean.png"), dpi=200)
plt.close()

# =========================
# 2) GCard 值与真实值的分布
# =========================

# 2b) 散点图：True vs GCard(1/2/3/mean)，对数坐标 + y=x
plt.figure()
true = to_num(df[TRUE_COL])
gmean = to_num(df["gcard_est_mean"])
plt.scatter(true, gmean, label="GCard mean", alpha=0.8)

for c in GCARD_EST_COLS:
    est = to_num(df[c])
    mask = est > 0
    plt.scatter(true[mask], to_num(est[mask]), alpha=0.35, label=c)

# y=x 参考线（在正数范围内）
true_pos = true.where(true > 0)
minv = np.nanmin(true_pos) if np.isfinite(np.nanmin(true_pos)) else 1
maxv = np.nanmax(true_pos) if np.isfinite(np.nanmax(true_pos)) else 10
xs = np.logspace(np.log10(minv), np.log10(maxv), 200)
plt.plot(xs, xs, linewidth=1)

plt.xscale("log")
plt.yscale("log")
plt.xlabel("True cardinality (log)")
plt.ylabel("Estimated cardinality (log)")
plt.title("True vs GCard Estimates (log-log)")
plt.legend(ncol=2, fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "2b_scatter_true_vs_gcard.png"), dpi=200)
plt.close()
# =========================
# 4) index 分布：饼状图（小类合并到 Others）
# =========================
idx = to_num(df[INDEX_COL]).dropna()
counts = idx.value_counts()
total = counts.sum()

min_ratio = 0.03  # 单类占比 < 3% 合并到 Others，可调
keep = counts[counts / total >= min_ratio]
others = counts[counts / total < min_ratio].sum()

pie_counts = keep.copy()
if others > 0:
    pie_counts.loc["Others"] = others

plt.figure(figsize=(7, 7))
plt.pie(
    pie_counts.values,
    labels=pie_counts.index.astype(str),
    autopct=lambda p: f"{p:.1f}%" if p >= 3 else "",
    startangle=90
)
plt.title(f"Distribution of index (Others < {min_ratio*100:.0f}%)")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "4_index_distribution_pie.png"), dpi=200)
plt.close()

# =========================
# Done
# =========================
print(f"Done. Figures saved to: {OUT_DIR}/")
for fn in sorted(os.listdir(OUT_DIR)):
    print(" -", fn)


/var/folders/9z/w1fgdq0j5637lz__9pv9zr2m0000gn/T/ipykernel_1841/827481465.py:50: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  plt.boxplot(data, labels=["SafeBound qerror", "GCard(mean of 1~3) qerror"], showfliers=True)


Done. Figures saved to: figs/
 - 1_qerror_boxplot_safebound_vs_gcard_mean.png
 - 2b_scatter_true_vs_gcard.png
 - 4_index_distribution_pie.png
